In [22]:
#Import the necessary Libraries
import numpy as np
import pandas as pd
import os
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings("ignore")

In [3]:
#Read cleaned csv file
df=pd.read_csv('cleaned_insurance_data.csv')
df.head()

,policy_id,subscription_length,vehicle_age,customer_age,region_code,region_density,segment,model,fuel_type,max_torque,...,is_brake_assist,is_power_door_locks,is_central_locking,is_power_steering,is_driver_seat_height_adjustable,is_day_night_rear_view_mirror,is_ecw,is_speed_alert,ncap_rating,claim_status
0,POL045360,9.3,1.2,41.0,C8,8794.0,C2,M4,Diesel,250Nm@2750rpm,...,Yes,Yes,Yes,Yes,Yes,No,Yes,Yes,3,0
1,POL016745,8.2,1.8,35.0,C2,27003.0,C1,M9,Diesel,200Nm@1750rpm,...,No,Yes,Yes,Yes,Yes,Yes,Yes,Yes,4,0
2,POL007194,9.5,0.2,44.0,C8,8794.0,C2,M4,Diesel,250Nm@2750rpm,...,Yes,Yes,Yes,Yes,Yes,No,Yes,Yes,3,0
3,POL018146,5.2,0.4,44.0,C10,58339.5,A,M1,CNG,60Nm@3500rpm,...,No,No,No,Yes,No,No,No,Yes,0,0
4,POL049011,10.1,1.0,56.0,C13,5410.0,B2,M5,Diesel,200Nm@3000rpm,...,No,Yes,Yes,Yes,No,No,Yes,Yes,5,0


In [4]:
#Dropped an irrelevant column (policy_id)
df.drop(columns=["policy_id"], inplace=True)
df

,subscription_length,vehicle_age,customer_age,region_code,region_density,segment,model,fuel_type,max_torque,max_power,...,is_brake_assist,is_power_door_locks,is_central_locking,is_power_steering,is_driver_seat_height_adjustable,is_day_night_rear_view_mirror,is_ecw,is_speed_alert,ncap_rating,claim_status
0,9.3,1.2,41.0,C8,8794.0,C2,M4,Diesel,250Nm@2750rpm,113.45bhp@4000rpm,...,Yes,Yes,Yes,Yes,Yes,No,Yes,Yes,3,0
1,8.2,1.8,35.0,C2,27003.0,C1,M9,Diesel,200Nm@1750rpm,97.89bhp@3600rpm,...,No,Yes,Yes,Yes,Yes,Yes,Yes,Yes,4,0
2,9.5,0.2,44.0,C8,8794.0,C2,M4,Diesel,250Nm@2750rpm,113.45bhp@4000rpm,...,Yes,Yes,Yes,Yes,Yes,No,Yes,Yes,3,0
3,5.2,0.4,44.0,C10,58339.5,A,M1,CNG,60Nm@3500rpm,40.36bhp@6000rpm,...,No,No,No,Yes,No,No,No,Yes,0,0
4,10.1,1.0,56.0,C13,5410.0,B2,M5,Diesel,200Nm@3000rpm,88.77bhp@4000rpm,...,No,Yes,Yes,Yes,No,No,Yes,Yes,5,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58587,10.6,2.6,48.0,C5,34738.0,B2,M6,Petrol,113Nm@4400rpm,88.50bhp@6000rpm,...,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,2,0
58588,2.3,2.2,37.0,C3,4076.0,C2,M4,Diesel,250Nm@2750rpm,113.45bhp@4000rpm,...,Yes,Yes,Yes,Yes,Yes,No,Yes,Yes,3,0
58589,6.6,2.2,35.0,C8,8794.0,B2,M6,Petrol,113Nm@4400rpm,88.50bhp@6000rpm,...,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,2,0
58590,4.1,3.6,44.0,C8,8794.0,B2,M6,Petrol,113Nm@4400rpm,88.50bhp@6000rpm,...,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,2,0


In [5]:
#Conversion of binary feature to numerical
binary_cols = df.select_dtypes(include="object").columns

for col in binary_cols:
    if df[col].nunique() == 2:
        df[col] = df[col].map({"Yes": 1, "No": 0})

In [8]:
df.head()

,subscription_length,vehicle_age,customer_age,region_code,region_density,segment,model,fuel_type,max_torque,max_power,...,is_brake_assist,is_power_door_locks,is_central_locking,is_power_steering,is_driver_seat_height_adjustable,is_day_night_rear_view_mirror,is_ecw,is_speed_alert,ncap_rating,claim_status
0,9.3,1.2,41.0,C8,8794.0,C2,M4,Diesel,250Nm@2750rpm,113.45bhp@4000rpm,...,1,1,1,1,1,0,1,1,3,0
1,8.2,1.8,35.0,C2,27003.0,C1,M9,Diesel,200Nm@1750rpm,97.89bhp@3600rpm,...,0,1,1,1,1,1,1,1,4,0
2,9.5,0.2,44.0,C8,8794.0,C2,M4,Diesel,250Nm@2750rpm,113.45bhp@4000rpm,...,1,1,1,1,1,0,1,1,3,0
3,5.2,0.4,44.0,C10,58339.5,A,M1,CNG,60Nm@3500rpm,40.36bhp@6000rpm,...,0,0,0,1,0,0,0,1,0,0
4,10.1,1.0,56.0,C13,5410.0,B2,M5,Diesel,200Nm@3000rpm,88.77bhp@4000rpm,...,0,1,1,1,0,0,1,1,5,0


In [9]:
#Conversion of categorical variables
df = pd.get_dummies(df, drop_first=True)
df.head()

,subscription_length,vehicle_age,customer_age,region_density,airbags,is_esc,is_adjustable_steering,is_tpms,is_parking_sensors,is_parking_camera,...,engine_type_1.5 L U2 CRDi,engine_type_1.5 Turbocharged Revotorq,engine_type_1.5 Turbocharged Revotron,engine_type_F8D Petrol Engine,engine_type_G12B,engine_type_K Series Dual jet,engine_type_K10C,engine_type_i-DTEC,steering_type_Manual,steering_type_Power
0,9.3,1.2,41.0,8794.0,6,1,1,1,1,1,...,1,0,0,0,0,0,0,0,0,1
1,8.2,1.8,35.0,27003.0,2,0,1,0,1,1,...,0,0,0,0,0,0,0,1,0,0
2,9.5,0.2,44.0,8794.0,6,1,1,1,1,1,...,1,0,0,0,0,0,0,0,0,1
3,5.2,0.4,44.0,58339.5,2,0,0,0,1,0,...,0,0,0,1,0,0,0,0,0,1
4,10.1,1.0,56.0,5410.0,2,0,1,0,1,0,...,0,1,0,0,0,0,0,0,0,0


### Feature Engineering

Feature engineering was performed to enhance model performance by incorporating domain-specific insights. New features such as vehicle risk score, safety score, and high-density indicators were created to better capture insurance risk patterns. 

Categorical variables were encoded, and highly correlated variables were removed to reduce multicollinearity.

In [10]:
#Vehicle Risk score
df["vehicle_risk_score"] = df["vehicle_age"] * df["displacement"]
df["vehicle_risk_score"]

0        1791.6
1        2696.4
2         298.6
3         318.4
4        1497.0
          ...  
58587    3112.2
58588    3284.6
58589    2633.4
58590    4309.2
58591     478.8
Name: vehicle_risk_score, Length: 58592, dtype: float64

In [11]:
#Safety Score
safety_cols = ["airbags", "is_esc", "is_tpms", "is_brake_assist"]

df["safety_score"] = df[safety_cols].sum(axis=1)
df["safety_score"]

0        9
1        2
2        9
3        2
4        2
        ..
58587    3
58588    9
58589    3
58590    3
58591    3
Name: safety_score, Length: 58592, dtype: int64

In [13]:
#Age group
df["age_group"] = pd.cut(
    df["customer_age"],
    bins=[18, 30, 50, 80],
    labels=["Young", "Adult", "Senior"]
)

df = pd.get_dummies(df, columns=["age_group"], drop_first=True)
df

,subscription_length,vehicle_age,customer_age,region_density,airbags,is_esc,is_adjustable_steering,is_tpms,is_parking_sensors,is_parking_camera,...,engine_type_K10C,engine_type_i-DTEC,steering_type_Manual,steering_type_Power,vehicle_risk_score,safety_score,age_group_Adult,age_group_Senior,age_group_Adult,age_group_Senior
0,9.3,1.2,41.0,8794.0,6,1,1,1,1,1,...,0,0,0,1,1791.6,9,1,0,1,0
1,8.2,1.8,35.0,27003.0,2,0,1,0,1,1,...,0,1,0,0,2696.4,2,1,0,1,0
2,9.5,0.2,44.0,8794.0,6,1,1,1,1,1,...,0,0,0,1,298.6,9,1,0,1,0
3,5.2,0.4,44.0,58339.5,2,0,0,0,1,0,...,0,0,0,1,318.4,2,1,0,1,0
4,10.1,1.0,56.0,5410.0,2,0,1,0,1,0,...,0,0,0,0,1497.0,2,0,1,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58587,10.6,2.6,48.0,34738.0,2,0,1,0,1,0,...,0,0,0,0,3112.2,3,1,0,1,0
58588,2.3,2.2,37.0,4076.0,6,1,1,1,1,1,...,0,0,0,1,3284.6,9,1,0,1,0
58589,6.6,2.2,35.0,8794.0,2,0,1,0,1,0,...,0,0,0,0,2633.4,3,1,0,1,0
58590,4.1,3.6,44.0,8794.0,2,0,1,0,1,0,...,0,0,0,0,4309.2,3,1,0,1,0


In [14]:
#Density Risk
df["high_density_area"] = (df["region_density"] > df["region_density"].median()).astype(int)
df

,subscription_length,vehicle_age,customer_age,region_density,airbags,is_esc,is_adjustable_steering,is_tpms,is_parking_sensors,is_parking_camera,...,engine_type_i-DTEC,steering_type_Manual,steering_type_Power,vehicle_risk_score,safety_score,age_group_Adult,age_group_Senior,age_group_Adult,age_group_Senior,high_density_area
0,9.3,1.2,41.0,8794.0,6,1,1,1,1,1,...,0,0,1,1791.6,9,1,0,1,0,0
1,8.2,1.8,35.0,27003.0,2,0,1,0,1,1,...,1,0,0,2696.4,2,1,0,1,0,1
2,9.5,0.2,44.0,8794.0,6,1,1,1,1,1,...,0,0,1,298.6,9,1,0,1,0,0
3,5.2,0.4,44.0,58339.5,2,0,0,0,1,0,...,0,0,1,318.4,2,1,0,1,0,1
4,10.1,1.0,56.0,5410.0,2,0,1,0,1,0,...,0,0,0,1497.0,2,0,1,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58587,10.6,2.6,48.0,34738.0,2,0,1,0,1,0,...,0,0,0,3112.2,3,1,0,1,0,1
58588,2.3,2.2,37.0,4076.0,6,1,1,1,1,1,...,0,0,1,3284.6,9,1,0,1,0,0
58589,6.6,2.2,35.0,8794.0,2,0,1,0,1,0,...,0,0,0,2633.4,3,1,0,1,0,0
58590,4.1,3.6,44.0,8794.0,2,0,1,0,1,0,...,0,0,0,4309.2,3,1,0,1,0,0


In [15]:
#Handled multicollinearity
drop_cols = ["length", "width", "turning_radius", "cylinder"]

df.drop(columns=drop_cols, inplace=True)

In [16]:
#Split features and Target
X = df.drop("claim_status", axis=1)
y = df["claim_status"]

In [17]:
#Train test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [18]:
#Feature scaling for Logistic regression
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [20]:
#Saved essential datasets
os.makedirs("../data", exist_ok=True)

# Save datasets
X_train.to_csv("../data/X_train.csv", index=False)
X_test.to_csv("../data/X_test.csv", index=False)
y_train.to_csv("../data/y_train.csv", index=False)
y_test.to_csv("../data/y_test.csv", index=False)

# Save scaler
joblib.dump(scaler, "../data/scaler.pkl")

print("✅ Essential files saved")

✅ Essential files saved
